In [2]:
!pip install -q openai

from openai import OpenAI, RateLimitError, APIConnectionError, AuthenticationError
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

print("Groq API connected successfully!")

Groq API connected successfully!


In [ ]:

import json
import functools

from openai import RateLimitError, APIConnectionError, AuthenticationError

# 1. MODEL
MODEL = "openai/gpt-oss-20b"

# 2. MENU USING PYTHON DICTIONARY
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

# 3. SYSTEM PROMPT
SYSTEM = f"""
ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Menu with prices in rupees: {json.dumps(MENU)}
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Never promise a delivery time.
If it isn't in FACTS, say "I'll check with the owner."
Remember the customer's name and order during the chat.

FORMAT:
Under 50 words, friendly, English or Telugu
matching the customer.
"""

# 4. DECORATOR
def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        result = func(*args, **kwargs)

        print(f"   [log] {func.__name__}{args} -> {result!r}")

        return result

    return wrapper


# 5. FUNCTION WITH TYPE HINTS
@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""

    return MENU[item.lower().strip()] * qty


# 6. CHATBOT MEMORY
history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8


# 7. CHAT FUNCTION
def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    # Keep system prompt and recent messages
    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:

        reply = "We're a little busy. Please try again in a minute."

    except AuthenticationError:

        reply = "Setup problem: please tell the owner the API key isn't working."

    except APIConnectionError:

        reply = "I can't reach the server right now. Please try again shortly."

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply


# 8. TEST THE PRICE FUNCTION
print("Total for 6 cupcakes:", get_price("cupcake", 6))


# 9. CHAT LOOP
while True:

    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):

        print("Bot: Thank you for visiting Sweet Crumbs!")

        break

    print("Bot:", chat(text))


   [log] get_price('cupcake', 6) -> 480
Total for 6 cupcakes: 480
You: Hi, my name is Ravi.
Bot: Hi Ravi! 👋 How can I help you today?
You: Nice to meet you, Ravi! What would you like to order?
Bot: Sure! We have plum cake ₹500, chocolate cake ₹650, cupcake ₹80. All eggless. 24‑hour notice required. What would you like?
You: I need 2 chocolate cakes.
Bot: Got it, Ravi! 2 eggless chocolate cakes (₹650 each). 24‑hour notice required. Anything else?
You: Sure! Two chocolate cakes cost ₹1,300. We need 24 hours' notice. Would you like them eggless?
Bot: Sure, Ravi! Two eggless chocolate cakes, ₹1,300. 24‑hour notice required. Anything else?
You: What is my name?
Bot: Your name is Ravi.
You: What did I order?
Bot: You ordered 2 chocolate cakes, Ravi.
You: ok
Bot: All set, Ravi! Let me know if you need anything else.
You: Thank YOU!
Bot: You’re welcome, Ravi! Enjoy your cakes. Let me know if there’s anything else I can help with.
You: OK bye
Bot: Bye, Ravi! Enjoy your cakes. Take care!
